# Peek at KO/EC links — annotation rows ↔ biosamples / studies

Worked-example queries showing how to navigate from a row in
`nmdc_results.annotation_kegg_orthology` (or `annotation_enzyme_commission`)
out to the biosample, study, and file metadata in `nmdc_metadata`.

**Anchor columns on the annotation tables:**

| column | meaning | join target |
|---|---|---|
| `workflow_run_id` | `nmdc:wfmgan-...` MetagenomeAnnotation activity | `nmdc_metadata.workflow_execution_set.id` |
| `data_object_id`  | `nmdc:dobj-...` source TSV file | `nmdc_metadata.data_object_set.id` |
| `gene_id`         | `<workflow_run_id>_<contig>_<start>_<end>` | local; matches functional GFFs (issues #84, #86–#90) |
| `ncbi_taxid`      | NCBI taxon for the BLAST hit | external (no native BERDL taxonomy table) |
| `annotation_id`   | `KO:Kxxxxx` or `EC:n.n.n.n` | KEGG / EC ontologies |

**The chain:**

```
annotation_*.workflow_run_id
  → workflow_execution_set.id  (.was_informed_by is array of dgns IDs)
  → data_generation_set.id
    → data_generation_set_has_input.has_input            (biosample IDs)
    → data_generation_set_associated_studies.associated_studies
  → biosample_set.{env_*, geo_loc_name_*, depth, …}
```

### Session setup

In [1]:
spark = get_spark_session(app_name="peek_ko_ec_links", tenant_name="nmdc")
print(f"Spark version: {spark.version}")

conn = get_trino_connection()
print("Trino connection ready")

Spark version: 4.0.1
Trino connection ready


### Preflight: confirm the annotation tables are registered

In [2]:
for tbl in ("annotation_kegg_orthology", "annotation_enzyme_commission"):
    if spark.sql(f"SHOW TABLES IN nmdc_results LIKE '{tbl}'").count() == 0:
        raise RuntimeError(f"nmdc_results.{tbl} not registered — run ingest_ko_ec_annotations.ipynb first")
    print(f"OK: nmdc_results.{tbl}")

OK: nmdc_results.annotation_kegg_orthology
OK: nmdc_results.annotation_enzyme_commission


### 1. Pick an anchor row

Grab one annotation row to drive the rest of the joins. Same recipe works for EC.

In [3]:
anchor = (
    spark.sql("""
        SELECT gene_id, annotation_id, workflow_run_id, data_object_id, ncbi_taxid
        FROM nmdc_results.annotation_kegg_orthology
        LIMIT 1
    """)
    .toPandas()
    .iloc[0]
)
WORKFLOW_RUN_ID = anchor["workflow_run_id"]
DATA_OBJECT_ID  = anchor["data_object_id"]
ANNOTATION_ID   = anchor["annotation_id"]
print(anchor.to_string())

gene_id            nmdc:wfmgan-11-4b91dx77.1_2431468_82_447
annotation_id                                     KO:K03752
workflow_run_id                   nmdc:wfmgan-11-4b91dx77.1
data_object_id                        nmdc:dobj-11-r4jvp282
ncbi_taxid                                       2995780880


### 2. `workflow_run_id` → workflow_execution_set

`was_informed_by` is **array-typed** (a workflow run can be informed by
multiple data generations). Use `array_contains` or `EXPLODE` to join through
it — equality won't work.

In [4]:
spark.sql(f"""
    SELECT id, type, was_informed_by, started_at_time, ended_at_time
    FROM nmdc_metadata.workflow_execution_set
    WHERE id = '{WORKFLOW_RUN_ID}'
""").toPandas()

,id,type,was_informed_by,started_at_time,ended_at_time
0,nmdc:wfmgan-11-4b91dx77.1,nmdc:MetagenomeAnnotation,[nmdc:dgns-11-e12jgq19],2025-07-14 20:38:51,2025-07-26 20:48:43


### 3. workflow_execution → data_generation_set (the sequencing run)

In [5]:
spark.sql(f"""
    SELECT dg.id, dg.type, dg.name, dg.processing_institution,
           dg.principal_investigator_name, dg.start_date, dg.end_date
    FROM (
        SELECT EXPLODE(was_informed_by) AS dgns_id
        FROM nmdc_metadata.workflow_execution_set
        WHERE id = '{WORKFLOW_RUN_ID}'
    ) info
    JOIN nmdc_metadata.data_generation_set dg
      ON dg.id = info.dgns_id
""").toPandas()


,id,type,name,processing_institution,principal_investigator_name,start_date,end_date
0,nmdc:dgns-11-e12jgq19,nmdc:NucleotideSequencing,Forest soil microbial communities from Barre W...,JGI,Jeffrey Blanchard,NaN,NaN


### Aside: NMDC's bipartite provenance graph

NMDC's data model has two parallel chains, bridged by `data_generation`:

```
biosample → material_processing → procsm → material_processing → procsm → ...   (material side)
                                                                          ↓
                                                                  data_generation   (bridge)
                                                                          ↓
data_object ← workflow_execution ← data_object ← workflow_execution ← ...        (data side)
```

To walk from an annotation row back to its source biosample(s) we need to
traverse both chains upstream. Spark 4.0.1 has a planner bug that prevents
`WITH RECURSIVE` in both Spark Connect and classic modes (`No plan for UnionLoop`).
Trino (also available in BERDL via `get_trino_connection()`) supports
`WITH RECURSIVE` natively and reads the same Delta tables.

**The four edge types used in the recursive walk:**

| Source table | src column | dst column | meaning |
|---|---|---|---|
| `workflow_execution_set_was_informed_by` | `parent_id` | `was_informed_by` | workflow run → data generation |
| `data_generation_set_has_input` | `parent_id` | `has_input` | data generation → biosample or procsm |
| `material_processing_set_has_output` | `has_output` | `parent_id` | procsm → the processing that made it |
| `material_processing_set_has_input` | `parent_id` | `has_input` | processing → its input (biosample or procsm) |

These four tables cover the full upstream walk regardless of chain depth.
The next cell defines helpers that run the `WITH RECURSIVE` query via Trino.

In [ ]:
import pandas as pd

_EDGES = """
    SELECT parent_id        AS src, was_informed_by AS next_id
    FROM   nmdc_metadata.workflow_execution_set_was_informed_by
    UNION ALL
    SELECT parent_id        AS src, has_input       AS next_id
    FROM   nmdc_metadata.data_generation_set_has_input
    UNION ALL
    SELECT has_output       AS src, parent_id       AS next_id
    FROM   nmdc_metadata.material_processing_set_has_output
    UNION ALL
    SELECT parent_id        AS src, has_input       AS next_id
    FROM   nmdc_metadata.material_processing_set_has_input
"""


def workflows_to_biosamples_bulk(conn, workflow_run_ids, max_depth: int = 15) -> pd.DataFrame:
    """Walk a list of workflow run IDs upstream to their source biosamples via Trino.

    Returns pd.DataFrame(workflow_run_id, biosample_id).
    One recursive CTE query; cost is independent of the number of IDs.
    """
    ids = list(set(workflow_run_ids))
    if not ids:
        return pd.DataFrame(columns=["workflow_run_id", "biosample_id"])

    values = ",\n        ".join(f"('{wid}')" for wid in ids)
    cur = conn.cursor()
    cur.execute(f"""
        WITH RECURSIVE upstream(origin, id, depth) AS (
            SELECT CAST(id AS VARCHAR) AS origin, CAST(id AS VARCHAR) AS id, CAST(0 AS BIGINT) AS depth
            FROM (VALUES
                {values}
            ) AS t(id)
            UNION ALL
            SELECT u.origin, e.next_id, u.depth + 1
            FROM   upstream u
            JOIN (
                {_EDGES}
            ) e ON e.src = u.id
            WHERE  u.depth < {max_depth}
              AND  u.id NOT LIKE 'nmdc:bsm%'
        )
        SELECT DISTINCT origin AS workflow_run_id, id AS biosample_id
        FROM   upstream
        WHERE  id LIKE 'nmdc:bsm%'
    """)
    rows = cur.fetchall()
    if not rows:
        return pd.DataFrame(columns=["workflow_run_id", "biosample_id"])
    return pd.DataFrame(rows, columns=["workflow_run_id", "biosample_id"])


def workflow_to_biosamples(conn, workflow_run_id: str, max_depth: int = 15) -> pd.DataFrame:
    """Single-workflow convenience wrapper. Returns biosample metadata rows."""
    bs = workflows_to_biosamples_bulk(conn, [workflow_run_id], max_depth)
    if bs.empty:
        return bs
    ids = ", ".join(f"'{i}'" for i in bs["biosample_id"])
    cur = conn.cursor()
    cur.execute(f"""
        SELECT id AS biosample_id, name,
               env_broad_scale_term_id, env_local_scale_term_id, env_medium_term_id,
               geo_loc_name_has_raw_value, depth_has_numeric_value, depth_has_unit
        FROM   nmdc_metadata.biosample_set
        WHERE  id IN ({ids})
    """)
    cols = [d[0] for d in cur.description]
    meta = pd.DataFrame(cur.fetchall(), columns=cols)
    return bs.merge(meta, on="biosample_id").drop(columns=["workflow_run_id"]).drop_duplicates().reset_index(drop=True)


### 4. workflow_execution → biosample(s) via the Trino recursive walk

`data_generation_set_has_input.has_input` may be a biosample directly *or* a
`processed_sample` (procsm) that must be chased back through
`material_processing_set` (Extraction / LibraryPreparation / Pooling / …).
The Trino `WITH RECURSIVE` CTE handles any depth automatically.

In our KO table: ~37.5 % of workflow runs reach a biosample in 2 hops;
~62.5 % go through a procsm chain (typically 8 hops: dgns → procsm →
LibraryPrep → procsm → Extraction → procsm → Pooling → 1–N biosamples).

In [7]:
workflow_to_biosamples(conn, WORKFLOW_RUN_ID)


TrinoUserError: TrinoUserError(type=USER_ERROR, name=TYPE_MISMATCH, message="line 8:13: recursion step relation output type (varchar) is not coercible to recursion base relation output type (varchar(25)) at column 2", query_id=20260430_164528_00030_extpw)

### 5. data_generation → study

In [ ]:
spark.sql(f"""
    SELECT s.id AS study_id, s.name, s.title, s.principal_investigator_name
    FROM (
        SELECT EXPLODE(was_informed_by) AS dgns_id
        FROM nmdc_metadata.workflow_execution_set
        WHERE id = '{WORKFLOW_RUN_ID}'
    ) info
    JOIN nmdc_metadata.data_generation_set_associated_studies dgs
      ON dgs.parent_id = info.dgns_id
    JOIN nmdc_metadata.study_set s
      ON s.id = dgs.associated_studies
""").toPandas()


### 6. `data_object_id` → data_object_set (file-level provenance)

Useful for verifying which source TSV a row came from, file size, MD5,
URL, etc. Not needed for downstream science.

In [ ]:
spark.sql(f"""
    SELECT id, name, data_object_type, file_size_bytes, md5_checksum, url
    FROM nmdc_metadata.data_object_set
    WHERE id = '{DATA_OBJECT_ID}'
""").toPandas()

### 7. Cross-check against `functional_annotation_agg`

`functional_annotation_agg` is a precomputed `(workflow_run, gene_function_id, count)`
rollup. Two caveats when joining to it:

1. **EC is absent** — `functional_annotation_agg` only carries `PFAM`,
   `KEGG.ORTHOLOGY`, and `COG`. The new `annotation_enzyme_commission` table
   is the only source of EC in BERDL.
2. **KO prefix differs** — `KO:Kxxxxx` here vs `KEGG.ORTHOLOGY:Kxxxxx` there.
   Translate before joining. The two values per workflow run *should* line up
   (`COUNT(*)` in `annotation_kegg_orthology` ≈ `SUM(count)` in the agg);
   any drift is worth flagging.

In [ ]:
spark.sql(f"""
    WITH ours AS (
        SELECT 'KEGG.ORTHOLOGY:' || SUBSTRING(annotation_id, 4) AS gene_function_id,
               COUNT(*) AS hits_in_ko_table
        FROM nmdc_results.annotation_kegg_orthology
        WHERE workflow_run_id = '{WORKFLOW_RUN_ID}'
        GROUP BY 1
    ),
    theirs AS (
        SELECT gene_function_id, count AS count_in_agg
        FROM nmdc_metadata.functional_annotation_agg
        WHERE was_generated_by = '{WORKFLOW_RUN_ID}'
          AND gene_function_id LIKE 'KEGG.ORTHOLOGY:%'
    )
    SELECT COALESCE(ours.gene_function_id, theirs.gene_function_id) AS gene_function_id,
           ours.hits_in_ko_table,
           theirs.count_in_agg,
           ours.hits_in_ko_table - theirs.count_in_agg AS diff
    FROM ours FULL OUTER JOIN theirs USING (gene_function_id)
    ORDER BY ABS(COALESCE(ours.hits_in_ko_table, 0) - COALESCE(theirs.count_in_agg, 0)) DESC
    LIMIT 20
""").toPandas()

### 8. End-to-end: KO hits per biosample for a chosen KO

The query that motivated the table — "how often does this KEGG Orthology
appear in each biosample?" — in a single Trino `WITH RECURSIVE` query.
Filter early on `annotation_id` so the heavy table is the smallest leg.

In [ ]:
import time

TARGET_KO = ANNOTATION_ID  # reuse the KO we anchored on; substitute any 'KO:Kxxxxx'

t0 = time.monotonic()
cur = conn.cursor()
cur.execute(f"""
    WITH
    ko_hits AS (
        SELECT workflow_run_id, COUNT(*) AS n_hits
        FROM   nmdc_results.annotation_kegg_orthology
        WHERE  annotation_id = '{TARGET_KO}'
        GROUP  BY workflow_run_id
    ),
    RECURSIVE upstream(origin, id, depth) AS (
        SELECT CAST(workflow_run_id AS VARCHAR) AS origin,
               CAST(workflow_run_id AS VARCHAR) AS id,
               CAST(0 AS BIGINT) AS depth
        FROM   ko_hits
        UNION ALL
        SELECT u.origin, e.next_id, u.depth + 1
        FROM   upstream u
        JOIN (
            {_EDGES}
        ) e ON e.src = u.id
        WHERE  u.depth < 15
          AND  u.id NOT LIKE 'nmdc:bsm%'
    ),
    run_to_biosample AS (
        SELECT DISTINCT origin AS workflow_run_id, id AS biosample_id
        FROM   upstream
        WHERE  id LIKE 'nmdc:bsm%'
    )
    SELECT bs.id                         AS biosample_id,
           bs.env_broad_scale_term_id,
           bs.env_medium_term_id,
           bs.geo_loc_name_has_raw_value,
           SUM(kh.n_hits)                AS total_hits
    FROM   run_to_biosample r2b
    JOIN   ko_hits kh ON kh.workflow_run_id = r2b.workflow_run_id
    JOIN   nmdc_metadata.biosample_set bs ON bs.id = r2b.biosample_id
    GROUP  BY 1, 2, 3, 4
    ORDER  BY total_hits DESC
    LIMIT  20
""")
cols = [d[0] for d in cur.description]
result = pd.DataFrame(cur.fetchall(), columns=cols)
print(f"({time.monotonic() - t0:.1f}s)")
result
